In [92]:
import numpy as np # type: ignore
import torch # type: ignore
import torch.nn as nn  # type: ignore
import torch.nn.functional as F # type: ignore
from torch.optim import AdamW
import math
import os
import sys
from pathlib import Path
import json
import requests # type: ignore
import glob


In [93]:
#getter

# ==========================================
# COSTANTI E CONFIGURAZIONI
# ==========================================

pokeUrl = "https://pokeapi.co/api/v2/"

types = {
    "normal": 1, "fighting": 2, "flying": 3, "poison": 4, "ground": 5,
    "rock": 6, "bug": 7, "ghost": 8, "steel": 9, "fire": 10, "water": 11,
    "grass": 12, "electric": 13, "psychic": 14, "ice": 15, "dragon": 16,
    "dark": 17, "fairy": 18
}

weather = {
    "none": 0, "SunnyDay": 1, "RainDance": 2, "Sandstorm": 3, "Snowscape": 4
}

# all_status = {
#     'par': 0, 'slp': 1, 'frz': 2, 'brn': 3, 'psn': 4, 'confusion': 5,
#     'infatuation': 6, 'trap': 7, 'nightmare': 8, 'torment': 9, 'disable': 10,
#     'yawn': 11, 'heal-block': 12, 'no-type-immunity': 13, 'leech-seed': 14,
#     'embargo': 15, 'perish-song': 16, 'ingrain': 17, 'Encore': 18, 'tox': 19
# }

all_status = {
    'par': 0, 'slp': 1, 'frz': 2, 'brn': 3, 'psn': 4, 'tox': 5
}

stat_code = {
    'atk': 0, 'def': 1, 'spa': 2, 'spd': 3, 'spe': 4
}

replacement = {
    'Aegislash': 'Aegislash-shield',
    'Maushold-Four': 'maushold-family-of-four',
    'Lycanroc': 'Lycanroc-Midday',
    'Sinistcha-Masterpiece': 'Sinistcha',
    'Maushold': 'maushold-family-of-three',
    'Mimikyu': '778',
    'Morpeko': '877',
    'Palafin': 'palafin-zero',
    'Basculegion-F': 'Basculegion-female',
    'Tauros-Paldea-Combat':'10250',
    'Tauros-Paldea-Blaze':'10251',
    'Tauros-Paldea-Aqua':'10252',
    'Meowstic-M-Mega':'Meowstic-male-Mega',
    'Basculegion':'Basculegion-male',
    'Basculegion-F':'Basculegion-female',
    'Pyroar':'Pyroar-male',
    "Polteageist-Antique":"855",
    "Polteageist-Antique":"855",
}

just_begin = {'Vivillon','Florges','Alcremie'}

to_set_sex = {'Basculegion', 'Meowstic'}

# ==========================================
# GESTIONE CACHE GLOBALE E HELPER API
# ==========================================

CACHE_FILE = "../data/"
os.makedirs(CACHE_FILE, exist_ok=True) # Crea la cartella se non esiste

def load_cache(which):
    file_path = CACHE_FILE + which + '_cache.json'
    if os.path.exists(file_path):
        try:
            with open(file_path, "r", encoding="utf-8") as f: # <-- Usa file_path qui!
                return json.load(f)
        except json.JSONDecodeError:
            pass # Se il file è corrotto o vuoto, partiamo da zero
    return {}

# Carichiamo la cache all'avvio del programma

_item_cache = load_cache('item')
_ability_cache = load_cache('ability')

def save_cache(which,cache):
    """Salva l'intero stato della cache nel file JSON."""
    with open(CACHE_FILE+which+'_cache.json', "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4) 

def get_cache_stats():
    return len(Pokemon._api_cache), len(Move._api_cache),len(_item_cache),len(_ability_cache)
   
def get_poke_data(targetUrl):
    try:
        risposta = requests.get(targetUrl)
        risposta.raise_for_status() 
        return risposta.json()
    except requests.exceptions.RequestException as e:
        print(f"Errore di connessione: {e}")
    except ValueError:
        print("La pagina non ha restituito un JSON valido.")
    return {}

def get_item_id(name):
    clean_name = name.replace(' ', '-')
    if clean_name in _item_cache:
        return _item_cache[clean_name]
    
    dati_json = get_poke_data(pokeUrl + 'item/' + clean_name)
    item_id = dati_json.get('id', 0)
    _item_cache[clean_name] = item_id
    save_cache('item', _item_cache) 
    return item_id

def get_ability_id(name):
    clean_name = name.replace(' ', '-')
    if clean_name in _ability_cache:
        return _ability_cache[clean_name]
    
    dati_json = get_poke_data(pokeUrl + 'ability/' + clean_name)
    ability_id = dati_json.get('id', 0)
    _ability_cache[clean_name] = ability_id
    save_cache('ability',_ability_cache) 
    return ability_id

def get_type(tyls):
    poke_types = [0, 0]
    for i, t in enumerate(tyls):
        poke_types[i] = types[t['type']['name']]
    return poke_types

def get_stat(poke_s):
    return [poke_s[i]['base_stat'] for i in range(6)]

def get_ability(abil):
    if len(abil) >= 1:
        return int(abil[0]['ability']['url'].split('/')[-2])
    return 0

# ==========================================
# CLASSI DEL GIOCO
# ==========================================

class Bitmask:
    def __init__(self, size, valore_iniziale=0):
        self.size = size
        self.LIMIT_MASK = (1 << size) - 1  
        self.mask = valore_iniziale & self.LIMIT_MASK

    def set_bit(self, indice, valore):
        if not (0 <= indice < self.size):
            raise IndexError(f"L'indice deve essere compreso tra 0 e {self.size - 1}.")
        if valore:
            self.mask |= (1 << indice)
        else:
            self.mask &= ~(1 << indice)

    def get_bit(self, indice):
        if not (0 <= indice < self.size):
            raise IndexError(f"L'indice deve essere compreso tra 0 e {self.size - 1}.")
        return bool((self.mask >> indice) & 1)

    def flip_bit(self, indice):
        if not (0 <= indice < self.size):
            raise IndexError(f"L'indice deve essere compreso tra 0 e {self.size - 1}.")
        self.mask ^= (1 << indice)

    def reset(self):
        self.mask = 0

    def to_list(self):
        return [self.get_bit(i) for i in range(self.size)]

    def to_tensor(self):
        arr_np = np.array(self.to_list(), dtype=np.float32)
        return torch.from_numpy(arr_np)

    def __str__(self):
        return f"{self.mask:0{self.size}b}"

    def __repr__(self):
        return str(self.mask)


class Battlefield:
    def __init__(self, turn, winner=-1, current_weather=0, speed_modifier=None):
        self.turn = turn
        self.winner = winner
        self.current_weather = current_weather
        
        if speed_modifier is None:
            self.speed_modifier = Bitmask(size=3)  
        elif isinstance(speed_modifier, Bitmask):
            self.speed_modifier = speed_modifier
        else:
            self.speed_modifier = Bitmask(size=3, valore_iniziale=speed_modifier)

    def to_list(self):
        return [self.turn, self.current_weather, self.speed_modifier.mask, self.winner]


class Pokemon:
    _api_cache = load_cache("pokemon") # Associa la cache alla variabile globale
    @staticmethod
    def fixname(name):
                
        if name in replacement: 
            return replacement[name]
        elif name.split('-')[0] in to_set_sex: 
            return name.split('-')[0] + "-male"  
        else:
            for el in just_begin:
                if name.startswith(el):
                    name = el

        name = name.replace('.','').replace(' ','-')
        return name
    
    def __init__(self, player, poke_id, stats_change=None, hp_ratio=1.0, slot=0, known_moves=None, item=0, status=None):
        self.player = int(player)
        self.name = poke_id
        
        fixed_name = self.fixname(poke_id)
        # Implementazione della Cache per Pokemon
        if fixed_name in Pokemon._api_cache:
            pokejs = Pokemon._api_cache[fixed_name]
        else:
            pokejs = get_poke_data(pokeUrl + "pokemon/" + fixed_name)
            Pokemon._api_cache[fixed_name] = pokejs
            save_cache('pokemon',Pokemon._api_cache)
        
        self.poke_id = pokejs['id']
        self.stats = get_stat(pokejs['stats'])
        self.types = get_type(pokejs['types'])
        self.ability = get_ability(pokejs['abilities'])
        
        self.stats_change = stats_change if stats_change is not None else [0, 0, 0, 0, 0]
        self.hp_ratio = hp_ratio 
        self.slot = slot

        
        self.known_moves = known_moves if known_moves is not None else [Move(0) for _ in range(4)]
        self.item = item
        
        if status is None:
            self.status = Bitmask(6)
        elif isinstance(status, Bitmask):
            self.status = status
        else:
            self.status = Bitmask(6, status)

    def to_list(self):
        moves_list = []
        for m in self.known_moves:
            moves_list += m.to_list()
        return [self.player, self.slot, self.poke_id] + self.types + [self.ability, self.item] + self.stats + self.stats_change + [self.status.mask] + moves_list + [self.hp_ratio]
    
    def add_move(self, new_move):
        for i in range(4):
            if self.known_moves[i].id == new_move.id:
                return 0
            if self.known_moves[i].id == 0: 
                self.known_moves[i] = new_move
                return 0 
        return -1
    
    def __str__(self):
        mosse_attive = [m for m in self.known_moves if m.id != 0]
        mosse_str = ", ".join(map(str, mosse_attive)) if mosse_attive else "Nessuna mossa"
        
        return (
            f"=== Pokémon ===\n"
            f"  ID/Nome      : {self.poke_id} ({self.name})\n"
            f"  Allenatore   : Giocatore {self.player} (Slot: {self.slot})\n"
            f"  Tipo         : {self.types} | Abilità: {self.ability}\n"
            f"  PS Residui   : {self.hp_ratio * 100:.1f}%\n"
            f"  Strumento    : {self.item}\n"
            f"  Stato Alter. : {self.status}\n"
            f"  Mosse Conos. : [{mosse_str}]\n"
            f"  Stat Modifier: {self.stats_change}\n"
            f"==============="
        )

    def __repr__(self):
        return f"Pokemon(Player: {self.player}, ID: {self.poke_id}, Nome: '{self.name}', HP: {self.hp_ratio * 100:.0f}%)"


class Move:
    _api_cache = load_cache("move")

    def __init__(self, move_name):
        if move_name == 0: 
            self.name = ''
            self.id = 0
            self.power = 0
            self.priority = 0
            self.type = 0
            self.d_class = 0
            self.t_class = 0
            self.accuracy = 0
        elif move_name == -1:
            self.name = 'switch'
            self.id = -1
            self.power = 0
            self.priority = 6
            self.type = 0
            self.d_class = 0
            self.t_class = 0
            self.accuracy = 0
        else:
            self.name = move_name
            clean_name = move_name.replace(' ', '-').replace("'", '')
            
            if clean_name in Move._api_cache:
                datajs = Move._api_cache[clean_name] 
            else:
                datajs = get_poke_data(pokeUrl + "move/" + clean_name)
                Move._api_cache[clean_name] = datajs 
                save_cache('move',Move._api_cache)
            
            self.d_class = int(datajs['damage_class']['url'].split('/')[-2])
            self.t_class = int(datajs['target']['url'].split('/')[-2])
            self.accuracy = datajs["accuracy"] if datajs["accuracy"] is not None else 100
            self.id = datajs['id']
            self.power = datajs['power'] if datajs['power'] is not None else 0
            self.priority = datajs['priority']
            self.type = types.get(datajs['type']['name'], 0)
    
    def to_list(self):
        return [self.id, self.type, self.d_class, self.t_class, self.accuracy, self.power, self.priority]

    def __str__(self):
        return f"{self.name}"
    
    def __repr__(self):
        return f"{self.name}"


class Action: 
    def __init__(self, usr_pl, usr_slot, trg_pl, trg_slot, move, mega=0):
        self.usr_slot = int(usr_slot)
        self.trg_slot = int(trg_slot)
        self.usr_pl = int(usr_pl)
        self.trg_pl = int(trg_pl)
        self.move = int(move)
        
        if self.move > 3 and self.move != 6:
            self.mega = 0
        else:
            self.mega = int(mega)

    def to_list(self):
        return [self.usr_pl, self.usr_slot, self.trg_pl, self.trg_slot, self.move, self.mega]

    def __str__(self):
        mega_str = " + Mega" if self.mega == 1 else ""
        return f"Act(from: p{self.usr_pl}{self.usr_slot} to: P{self.trg_pl}{self.trg_slot}; {self.move}){mega_str}"

    def __repr__(self):
        return f"Action({self.usr_pl}, {self.usr_slot}, {self.trg_pl}, {self.trg_slot}, {self.move}, {self.mega})"


In [94]:
#Legal action mask

src_path = Path('/kaggle/working/src') 

# Aggiungi il percorso al sys.path solo se non c'è già
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))



class ActionMasker:
    def __init__(self):
        # Le tue dimensioni: 2x2x2x5x2x6
        self.dims = (2, 2, 2, 5, 2, 6)
        self.total_actions = np.prod(self.dims) # 480

    def get_flat_index(self, p_user, s_user, p_target, s_target, mega, move):
        """Converte le 6 componenti in un singolo indice da 0 a 479"""
        return np.ravel_multi_index(
            (p_user, s_user, p_target, s_target, mega, move), 
            self.dims
        )

    def get_valid_action_mask(self, state, move, first_state):
        """
        Analizza lo stato corrente e restituisce una maschera booleana [192]
        dove True = Azione consentita, False = Azione illegale.
        """
        mega_disp = False
        if state['player'][0] == 0:
            current_poke_id = state['id'][:6]
            first_poke_id = first_state['id'][:6]
        else:
            current_poke_id = state['id'][6:12]
            first_poke_id = first_state['id'][6:12]
        
        if np.array_equal(current_poke_id, first_poke_id):
            mega_disp = True
        

        

        # Partiamo assumendo che nessuna azione sia valida
        mask = np.zeros(self.total_actions, dtype=bool)
        if np.array_equal(state['id'], np.zeros_like(state['id'])):
            mask = np.ones(self.total_actions, dtype=bool)

        # p_user = 0  Assumiamo che 0 sia il giocatore corrente (dimensione 1)
        mask[self.get_flat_index(0, 0, 0, 0, 0, 5)] = True
        mask[self.get_flat_index(0, 1, 0, 1, 0, 5)] = True

        # Controlliamo le azioni per ciascuno dei tuoi 2 Pokémon attivi
        for s_user in [1,2]: 
            
            for i in range(12):
                #se esiste vivo in 3 -> sw in 3 disp
                if state['slot'][i] == 3 and state['hp_ratio'][i] != 0:
                    mask[self.get_flat_index(0, s_user-1, 0, 3, 0, 4)] = True
                #se esiste vivo in 4 -> sw in 4 disp
                if state['slot'][i] == 4 and state['hp_ratio'][i] != 0:
                    mask[self.get_flat_index(0, s_user-1, 0, 4, 0, 4)] = True
                
            if not self.present_in(state,4):
                mask[self.get_flat_index(0, s_user-1, 0, 0, 0, 4)] = True
                
            if self.is_pokemon_alive(state, s_user):
                for mve in [0,1,2,3]:
                    for trg_pl in [0,1]:
                        for trg_sl in [1,2]:
                            for i in range(12):
                                if state['slot'][i] == s_user and state['player'][i] == 0 and move['id'][i][mve] != 0:
                                    if move['t_class'][i][mve] in [4,13,5] and trg_pl == 0: 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True
                                    elif move['t_class'][i][mve] in [3,11] and trg_pl == 1: 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True
                                    elif move['t_class'][i][mve] in [12,14,16]: 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True
                                    elif move['t_class'][i][mve] in [9, 1, 2, 8, 10] and (trg_pl != 0 or trg_sl != s_user): 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True
                                    elif move['t_class'][i][mve] in [3,15] and trg_pl == 0 and trg_sl != s_user: 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True
                                    elif move['t_class'][i][mve] in [7] and trg_pl == 0 and trg_sl == s_user: 
                                        mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 0, mve)] = True
                                        if mega_disp and state['item'][i] == 2177:  
                                            mask[self.get_flat_index(0, s_user-1, trg_pl, trg_sl, 1, mve)] = True

        return mask

    # --- METODI DUMMY (Da implementare con la logica del tuo simulatore) ---
    def is_pokemon_alive(self, state, slot):
        #poke slot = 0 -> slot sconosciuto 
        for i in range(12):
            if state['slot'][i] == slot and state['player'][i] == 0:
                if state['hp_ratio'][i][0] == 0.0:
                    return False
                else:
                    return True
        return False

    def present_in(self, state, slot):
        for i in range (12):
            if state['slot'][i] == slot: 
                return True
        return False 


In [96]:
#PokemonVGCDataset

from torch.utils.data import Dataset, DataLoader # type: ignore
#from LegalActionMask import ActionMasker  # adatta il path all'import reale


class PokemonVGCDataset(Dataset):
    def __init__(self, file_paths, max_turn=49):
        self.file_paths = file_paths
        self.max_turn = max_turn
        self.masker = ActionMasker()

        # NUOVO: Definisci e crea una cartella scrivibile per le maschere
        self.cache_dir = '/kaggle/working/mask_cache'
        os.makedirs(self.cache_dir, exist_ok=True)
    
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # 1. Caricamento del file .npz o .npy
        file_path = self.file_paths[idx]
        
        # Estrai solo il nome del file (es. 'gen9champions...npz')
        file_name = os.path.basename(file_path)
        
        # Crea il nuovo path in /kaggle/working/mask_cache/
        mask_cache_path = os.path.join(self.cache_dir, file_name.replace('.npz', '_legalmask.npy'))

        # Apriamo il file e gestiamo l'estrazione di 'turns' se è un archivio .npz
        with np.load(file_path, allow_pickle=True) as loaded_file:
            data = loaded_file['turns'] if 'turns' in loaded_file else loaded_file

        num_turns = min(len(data), self.max_turn) # Tronchiamo se supera max_turn
        
        # Inizializziamo i tensori vuoti con il padding (es. zeri)
        # Dimensioni target: (max_turn, 12, ...) per state/move, (max_turn, 2, ...) per action
        
        # --- PREPARAZIONE DIZIONARI VUOTI ---
        state = {
            'id': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'player': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'type1': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'type2': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'ability': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'item': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'slot': torch.zeros((self.max_turn, 12), dtype=torch.long),
            'stats': torch.zeros((self.max_turn, 12, 6), dtype=torch.float32), # cont
            'stats_change': torch.zeros((self.max_turn, 12, 5), dtype=torch.float32), # cont
            'status': torch.zeros((self.max_turn, 12, 6), dtype=torch.float32), # cont
            'hp_ratio': torch.zeros((self.max_turn, 12, 1), dtype=torch.float32) # cont
        }
        
        move = {
            'id': torch.zeros((self.max_turn, 12, 4), dtype=torch.long),
            'd_class': torch.zeros((self.max_turn, 12, 4), dtype=torch.long),
            't_class': torch.zeros((self.max_turn, 12, 4), dtype=torch.long),
            'type': torch.zeros((self.max_turn, 12, 4), dtype=torch.long),
            'power': torch.zeros((self.max_turn, 12, 4, 1), dtype=torch.float32), # cont
            'priority': torch.zeros((self.max_turn, 12, 4), dtype=torch.float32), 
            'accuracy': torch.zeros((self.max_turn, 12, 4), dtype=torch.float32) # cont
            
            
        }
        
        battlefield = {
            'current_weather': torch.zeros((self.max_turn,), dtype=torch.long),
            'speed_modifier': torch.zeros((self.max_turn, 3), dtype=torch.float32) 
        }
        
        action = {
            'player_user': torch.zeros((self.max_turn, 2), dtype=torch.long),
            'slot_user': torch.zeros((self.max_turn, 2), dtype=torch.long),
            'player_target': torch.zeros((self.max_turn, 2), dtype=torch.long),
            'slot_target': torch.zeros((self.max_turn, 2), dtype=torch.long),
            'mega': torch.zeros((self.max_turn, 2), dtype=torch.long),
            'move': torch.zeros((self.max_turn, 2), dtype=torch.long)
        }

        reward = torch.zeros((self.max_turn,), dtype=torch.long)
        turn_tensor = torch.arange(self.max_turn, dtype=torch.long)
        
        # Maschera per i turni validi
        padding_mask = torch.zeros((self.max_turn,), dtype=torch.long)
        padding_mask[:num_turns] = 1

        mask_cache_hit = os.path.exists(mask_cache_path)
        if mask_cache_hit:
            legal_action_mask = torch.from_numpy(np.load(mask_cache_path))
        else:
            legal_action_mask = torch.ones((self.max_turn, self.masker.total_actions), dtype=torch.bool)
            first_poke_data = data[0]['pokemon']
            first_state_np = {'id': np.array([int(first_poke_data[i]['poke_id']) for i in range(12)])}
            
        # 2. ESTRAZIONE DATI DAL NUMPY E POPOLAMENTO TENSORI
        for t in range(num_turns):
            turn_data = data[t]
            poke_data = turn_data['pokemon'] # Array di 12 pokemon
            
            # Popolamento Campo (Battlefield)
            battlefield['current_weather'][t] = int(turn_data['field']['weather'])
            battlefield['speed_modifier'][t] = torch.tensor((turn_data['field']['speed_mask'] >> np.arange(2, -1, -1)) & 1, dtype=torch.float32)

            # NOTA: speed_mask in numpy è uno scalare, ma l'embedding vuole 3 feature continue.
            # Dovrete scompattare speed_mask o modificare l'Embedding.
            
            # Popolamento Azioni (Stacking action0 e action1)
            for a_idx, act_key in enumerate(['action0', 'action1']):
                act_data = turn_data[act_key]
                action['player_user'][t, a_idx] = int(act_data['usr_pl'])
                action['slot_user'][t, a_idx] = int(act_data['usr_slot'])
                action['player_target'][t, a_idx] = int(act_data['trg_pl'])
                action['slot_target'][t, a_idx] = int(act_data['trg_slot'])
                action['mega'][t, a_idx] = int(act_data['mega'])
                action['move'][t, a_idx] = int(act_data['move'])

            # Popolamento Pokemon e Mosse
            for p in range(12):
                p_data = poke_data[p]
                
                # Features discrete
                state['player'][t, p] = int(p_data['player'])
                
                poke_id_str = str(int(p_data['poke_id']))
                state['id'][t, p] = pokemon_map.get(poke_id_str, 0)
                
                state['type1'][t, p] = int(p_data['type1']) 
                state['type2'][t, p] = int(p_data['type2']) 
                
                ability_str = str(int(p_data['ability']))
                state['ability'][t, p] = ability_map.get(ability_str, 0)
                
                item_str = str(int(p_data['item']))
                state['item'][t, p] = item_map.get(item_str, 0)
                
                state['slot'][t, p] = int(p_data['slot'])
                
                # Features continue (Raggruppiamo i campi scalari in array)
                state['stats'][t, p] = torch.tensor([p_data['hp_base'], p_data['atk'], p_data['def_'], p_data['spa'], p_data['spd'], p_data['spe']], dtype=torch.float32)/ 255.0
                state['stats_change'][t, p] = torch.tensor([p_data['atk_c'], p_data['def_c'], p_data['spa_c'], p_data['spd_c'], p_data['spe_c']], dtype=torch.float32)/ 6.0  
                state['hp_ratio'][t, p, 0] = float(p_data['hp_ratio'])
                state['status'][t, p] = torch.tensor((p_data['status_mask'] >> np.arange(5, -1, -1)) & 1, dtype=torch.float32)
                

                # Estrazione delle 4 mosse (Stacking move0, move1, move2, move3)
                for m_idx, move_key in enumerate(['move0', 'move1', 'move2', 'move3']):
                    m_data = p_data[move_key]
                    move_id_str = str(int(m_data['id']))
                    move['id'][t, p, m_idx] = move_map.get(move_id_str, 0)
                    move['d_class'][t, p, m_idx] = int(m_data['d_class'])
                    move['t_class'][t, p, m_idx] = int(m_data['t_class'])
                    move['power'][t, p, m_idx, 0] = float(m_data['power'])/255.0
                    move['priority'][t, p, m_idx] = float(m_data['priority'])/8.0
                    move['accuracy'][t, p, m_idx] = float(m_data['accuracy'])/100.0
                    move['type'][t, p, m_idx] = float(m_data['type'])

            # Esempio di gestione Reward (se winner è nel campo, assegnare 1 al turno finale o simile)
            if t == num_turns - 1: 
                 reward[t] = int(turn_data['field']['winner'])

            if not mask_cache_hit:
                state_np = {
                    'player':   np.array([int(poke_data[i]['player'])   for i in range(12)]),
                    'id':       np.array([int(poke_data[i]['poke_id'])  for i in range(12)]),
                    'slot':     np.array([int(poke_data[i]['slot'])     for i in range(12)]),
                    'item':     np.array([int(poke_data[i]['item'])     for i in range(12)]),
                    'hp_ratio': np.array([[float(poke_data[i]['hp_ratio'])] for i in range(12)]),
                }
                move_np = {
                    'id':      np.array([[int(poke_data[i][f'move{m}']['id'])      for m in range(4)] for i in range(12)]),
                    't_class': np.array([[int(poke_data[i][f'move{m}']['t_class']) for m in range(4)] for i in range(12)]),
                }
                mask_t = self.masker.get_valid_action_mask(state_np, move_np, first_state_np)
                legal_action_mask[t] = torch.from_numpy(mask_t)

        # Creazione del target_actions per la Loss (usando gli ID delle azioni scelte)
        if not mask_cache_hit:
            np.save(mask_cache_path, legal_action_mask.numpy())
        target_actions = {k: v.clone() for k, v in action.items()} 

        return {
            'state': state,
            'move': move,
            'battlefield': battlefield,
            'action': action,
            'reward': reward,
            'turn': turn_tensor,
            'padding_mask': padding_mask,
            'target_actions': target_actions,
            'legal_action_mask': legal_action_mask,   # <-- nuovo, shape (max_turn, 480)

        }


In [95]:
#mappe 


#272=pokemon reg M-A  
#310=pokemon reg M-B 
#lunghezza token 544


ability_map = {
        "0": 0,
        "1": 1,
        "2": 2,
        "3": 3,
        "5": 4,
        "7": 5,
        "8": 6,
        "9": 7,
        "10": 8,
        "11": 9,
        "12": 10,
        "13": 11,
        "14": 12,
        "15": 13,
        "17": 14,
        "18": 15,
        "19": 16,
        "20": 17,
        "22": 18,
        "23": 19,
        "24": 20,
        "26": 21,
        "28": 22,
        "29": 23,
        "30": 24,
        "31": 25,
        "33": 26,
        "34": 27,
        "35": 28,
        "36": 29,
        "37": 30,
        "39": 31,
        "40": 32,
        "44": 33,
        "45": 34,
        "46": 35,
        "47": 36,
        "48": 37,
        "49": 38,
        "51": 39,
        "52": 40,
        "53": 41,
        "56": 42,
        "62": 43,
        "63": 44,
        "65": 45,
        "66": 46,
        "67": 47,
        "68": 48,
        "69": 49,
        "70": 50,
        "73": 51,
        "74": 52,
        "75": 53,
        "77": 54,
        "80": 55,
        "81": 56,
        "84": 57,
        "86": 58,
        "87": 59,
        "89": 60,
        "91": 61,
        "92": 62,
        "94": 63,
        "99": 64,
        "101": 65,
        "102": 66,
        "104": 67,
        "107": 68,
        "111": 69,
        "113": 70,
        "115": 71,
        "117": 72,
        "119": 73,
        "124": 74,
        "125": 75,
        "127": 76,
        "128": 77,
        "130": 78,
        "131": 79,
        "132": 80,
        "133": 81,
        "134": 82,
        "136": 83,
        "140": 84,
        "141": 85,
        "142": 86,
        "143": 87,
        "144": 88,
        "146": 89,
        "149": 90,
        "151": 91,
        "152": 92,
        "153": 93,
        "154": 94,
        "156": 95,
        "157": 96,
        "158": 97,
        "159": 98,
        "166": 99,
        "168": 100,
        "171": 101,
        "172": 102,
        "173": 103,
        "174": 104,
        "175": 105,
        "176": 106,
        "177": 107,
        "178": 108,
        "180": 109,
        "181": 110,
        "182": 111,
        "184": 112,
        "185": 113,
        "187": 114,
        "192": 115,
        "196": 116,
        "199": 117,
        "201": 118,
        "204": 119,
        "209": 120,
        "212": 121,
        "214": 122,
        "215": 123,
        "222": 124,
        "240": 125,
        "242": 126,
        "245": 127,
        "247": 128,
        "254": 129,
        "258": 130,
        "259": 131,
        "260": 132,
        "261": 133,
        "272": 134,
        "278": 135,
        "280": 136,
        "290": 137,
        "291": 138,
        "292": 139,
        "295": 140,
        "296": 141,
        "297": 142,
        "300": 143,
        "301": 144,
        "308": 145,
        "309": 146,
        "310": 147,
        "311": 148
    }

move_map = {
        "0": 0,
        "7": 1,
        "8": 2,
        "9": 3,
        "14": 4,
        "18": 5,
        "25": 6,
        "34": 7,
        "38": 8,
        "42": 9,
        "46": 10,
        "50": 11,
        "53": 12,
        "56": 13,
        "57": 14,
        "58": 15,
        "59": 16,
        "63": 17,
        "65": 18,
        "67": 19,
        "73": 20,
        "76": 21,
        "79": 22,
        "83": 23,
        "85": 24,
        "86": 25,
        "87": 26,
        "89": 27,
        "90": 28,
        "92": 29,
        "94": 30,
        "95": 31,
        "98": 32,
        "101": 33,
        "105": 34,
        "107": 35,
        "113": 36,
        "114": 37,
        "115": 38,
        "116": 39,
        "120": 40,
        "126": 41,
        "127": 42,
        "136": 43,
        "137": 44,
        "141": 45,
        "153": 46,
        "156": 47,
        "157": 48,
        "162": 49,
        "164": 50,
        "165": 51,
        "174": 52,
        "182": 53,
        "183": 54,
        "187": 55,
        "188": 56,
        "194": 57,
        "195": 58,
        "196": 59,
        "197": 60,
        "198": 61,
        "200": 62,
        "202": 63,
        "203": 64,
        "204": 65,
        "207": 66,
        "219": 67,
        "226": 68,
        "227": 69,
        "235": 70,
        "236": 71,
        "238": 72,
        "240": 73,
        "241": 74,
        "242": 75,
        "243": 76,
        "244": 77,
        "245": 78,
        "246": 79,
        "247": 80,
        "250": 81,
        "251": 82,
        "252": 83,
        "254": 84,
        "257": 85,
        "261": 86,
        "263": 87,
        "266": 88,
        "269": 89,
        "270": 90,
        "271": 91,
        "276": 92,
        "280": 93,
        "281": 94,
        "282": 95,
        "283": 96,
        "284": 97,
        "285": 98,
        "286": 99,
        "298": 100,
        "303": 101,
        "304": 102,
        "307": 103,
        "308": 104,
        "309": 105,
        "311": 106,
        "313": 107,
        "315": 108,
        "317": 109,
        "322": 110,
        "323": 111,
        "326": 112,
        "328": 113,
        "329": 114,
        "330": 115,
        "331": 116,
        "333": 117,
        "334": 118,
        "336": 119,
        "337": 120,
        "339": 121,
        "344": 122,
        "347": 123,
        "348": 124,
        "349": 125,
        "350": 126,
        "352": 127,
        "355": 128,
        "356": 129,
        "359": 130,
        "361": 131,
        "364": 132,
        "366": 133,
        "369": 134,
        "370": 135,
        "372": 136,
        "374": 137,
        "387": 138,
        "388": 139,
        "389": 140,
        "394": 141,
        "396": 142,
        "398": 143,
        "399": 144,
        "400": 145,
        "403": 146,
        "404": 147,
        "405": 148,
        "406": 149,
        "408": 150,
        "409": 151,
        "410": 152,
        "411": 153,
        "412": 154,
        "413": 155,
        "414": 156,
        "416": 157,
        "417": 158,
        "418": 159,
        "420": 160,
        "421": 161,
        "423": 162,
        "425": 163,
        "427": 164,
        "428": 165,
        "430": 166,
        "433": 167,
        "434": 168,
        "435": 169,
        "437": 170,
        "438": 171,
        "441": 172,
        "442": 173,
        "444": 174,
        "446": 175,
        "450": 176,
        "452": 177,
        "453": 178,
        "457": 179,
        "469": 180,
        "473": 181,
        "476": 182,
        "479": 183,
        "480": 184,
        "482": 185,
        "483": 186,
        "484": 187,
        "487": 188,
        "488": 189,
        "489": 190,
        "490": 191,
        "491": 192,
        "492": 193,
        "493": 194,
        "494": 195,
        "495": 196,
        "496": 197,
        "500": 198,
        "501": 199,
        "502": 200,
        "503": 201,
        "504": 202,
        "505": 203,
        "506": 204,
        "511": 205,
        "512": 206,
        "515": 207,
        "521": 208,
        "522": 209,
        "523": 210,
        "525": 211,
        "527": 212,
        "528": 213,
        "529": 214,
        "533": 215,
        "534": 216,
        "535": 217,
        "538": 218,
        "542": 219,
        "552": 220,
        "555": 221,
        "556": 222,
        "566": 223,
        "570": 224,
        "572": 225,
        "573": 226,
        "575": 227,
        "577": 228,
        "580": 229,
        "583": 230,
        "585": 231,
        "588": 232,
        "594": 233,
        "595": 234,
        "596": 235,
        "598": 236,
        "605": 237,
        "608": 238,
        "611": 239,
        "617": 240,
        "661": 241,
        "663": 242,
        "665": 243,
        "667": 244,
        "668": 245,
        "675": 246,
        "676": 247,
        "678": 248,
        "679": 249,
        "682": 250,
        "683": 251,
        "688": 252,
        "689": 253,
        "691": 254,
        "694": 255,
        "706": 256,
        "707": 257,
        "709": 258,
        "710": 259,
        "751": 260,
        "775": 261,
        "776": 262,
        "777": 263,
        "783": 264,
        "784": 265,
        "787": 266,
        "791": 267,
        "796": 268,
        "797": 269,
        "799": 270,
        "801": 271,
        "802": 272,
        "803": 273,
        "807": 274,
        "808": 275,
        "809": 276,
        "811": 277,
        "812": 278,
        "813": 279,
        "814": 280,
        "815": 281,
        "826": 282,
        "827": 283,
        "828": 284,
        "830": 285,
        "834": 286,
        "836": 287,
        "838": 288,
        "841": 289,
        "842": 290,
        "843": 291,
        "845": 292,
        "854": 293,
        "855": 294,
        "857": 295,
        "858": 296,
        "860": 297,
        "861": 298,
        "864": 299,
        "866": 300,
        "869": 301,
        "870": 302,
        "871": 303,
        "872": 304,
        "873": 305,
        "880": 306,
        "881": 307,
        "883": 308,
        "886": 309,
        "888": 310,
        "890": 311,
        "891": 312,
        "893": 313,
        "895": 314,
        "902": 315,
        "905": 316,
        "907": 317,
        "913": 318,
        "915": 319,
        "917": 320,
        "918": 321
    }

pokemon_map = {
        "3": 0,
        "6": 1,
        "9": 2,
        "15": 3,
        "18": 4,
        "25": 5,
        "26": 6,
        "36": 7,
        "38": 8,
        "59": 9,
        "65": 10,
        "68": 11,
        "71": 12,
        "80": 13,
        "94": 14,
        "115": 15,
        "121": 16,
        "127": 17,
        "128": 18,
        "130": 19,
        "132": 20,
        "134": 21,
        "135": 22,
        "142": 23,
        "143": 24,
        "149": 25,
        "154": 26,
        "160": 27,
        "168": 28,
        "181": 29,
        "184": 30,
        "186": 31,
        "197": 32,
        "199": 33,
        "205": 34,
        "208": 35,
        "212": 36,
        "214": 37,
        "227": 38,
        "229": 39,
        "248": 40,
        "279": 41,
        "282": 42,
        "302": 43,
        "306": 44,
        "308": 45,
        "310": 46,
        "319": 47,
        "323": 48,
        "324": 49,
        "334": 50,
        "350": 51,
        "354": 52,
        "358": 53,
        "359": 54,
        "362": 55,
        "389": 56,
        "392": 57,
        "395": 58,
        "409": 59,
        "411": 60,
        "428": 61,
        "442": 62,
        "445": 63,
        "448": 64,
        "450": 65,
        "454": 66,
        "460": 67,
        "461": 68,
        "464": 69,
        "470": 70,
        "471": 71,
        "472": 72,
        "473": 73,
        "475": 74,
        "478": 75,
        "497": 76,
        "500": 77,
        "510": 78,
        "530": 79,
        "531": 80,
        "534": 81,
        "547": 82,
        "553": 83,
        "563": 84,
        "569": 85,
        "579": 86,
        "584": 87,
        "609": 88,
        "614": 89,
        "623": 90,
        "635": 91,
        "637": 92,
        "652": 93,
        "655": 94,
        "658": 95,
        "660": 96,
        "663": 97,
        "666": 98,
        "671": 99,
        "675": 100,
        "678": 101,
        "681": 102,
        "683": 103,
        "695": 104,
        "699": 105,
        "700": 106,
        "701": 107,
        "706": 108,
        "707": 109,
        "715": 110,
        "727": 111,
        "730": 112,
        "740": 113,
        "745": 114,
        "748": 115,
        "750": 116,
        "752": 117,
        "758": 118,
        "763": 119,
        "765": 120,
        "766": 121,
        "778": 122,
        "780": 123,
        "784": 124,
        "823": 125,
        "841": 126,
        "842": 127,
        "844": 128,
        "855": 129,
        "858": 130,
        "866": 131,
        "867": 132,
        "869": 133,
        "877": 134,
        "887": 135,
        "899": 136,
        "900": 137,
        "902": 138,
        "903": 139,
        "908": 140,
        "911": 141,
        "914": 142,
        "925": 143,
        "934": 144,
        "936": 145,
        "937": 146,
        "939": 147,
        "952": 148,
        "956": 149,
        "959": 150,
        "964": 151,
        "968": 152,
        "970": 153,
        "981": 154,
        "983": 155,
        "1013": 156,
        "1018": 157,
        "1019": 158,
        "10008": 159,
        "10009": 160,
        "10010": 161,
        "10012": 162,
        "10032": 163,
        "10033": 164,
        "10034": 165,
        "10035": 166,
        "10036": 167,
        "10037": 168,
        "10038": 169,
        "10039": 170,
        "10040": 171,
        "10041": 172,
        "10042": 173,
        "10045": 174,
        "10046": 175,
        "10047": 176,
        "10048": 177,
        "10049": 178,
        "10051": 179,
        "10053": 180,
        "10054": 181,
        "10055": 182,
        "10056": 183,
        "10057": 184,
        "10058": 185,
        "10059": 186,
        "10060": 187,
        "10061": 188,
        "10067": 189,
        "10068": 190,
        "10069": 191,
        "10070": 192,
        "10071": 193,
        "10072": 194,
        "10073": 195,
        "10074": 196,
        "10087": 197,
        "10088": 198,
        "10090": 199,
        "10104": 200,
        "10126": 201,
        "10143": 202,
        "10152": 203,
        "10165": 204,
        "10172": 205,
        "10230": 206,
        "10233": 207,
        "10236": 208,
        "10239": 209,
        "10242": 210,
        "10243": 211,
        "10244": 212,
        "10248": 213,
        "10251": 214,
        "10252": 215,
        "10256": 216,
        "10257": 217,
        "10278": 218,
        "10279": 219,
        "10280": 220,
        "10281": 221,
        "10282": 222,
        "10283": 223,
        "10284": 224,
        "10285": 225,
        "10286": 226,
        "10287": 227,
        "10291": 228,
        "10292": 229,
        "10293": 230,
        "10294": 231,
        "10296": 232,
        "10300": 233,
        "10302": 234,
        "10306": 235,
        "10313": 236,
        "10314": 237,
        "10315": 238,
        "10320": 239,
        "10321": 240
    }

item_map = {
        "0": 0,
        "127": 1,
        "134": 2,
        "135": 3,
        "161": 4,
        "162": 5,
        "163": 6,
        "164": 7,
        "165": 8,
        "166": 9,
        "167": 10,
        "168": 11,
        "169": 12,
        "172": 13,
        "173": 14,
        "174": 15,
        "175": 16,
        "176": 17,
        "190": 18,
        "191": 19,
        "196": 20,
        "209": 21,
        "211": 22,
        "214": 23,
        "216": 24,
        "217": 25,
        "218": 26,
        "219": 27,
        "220": 28,
        "223": 29,
        "224": 30,
        "226": 31,
        "227": 32,
        "230": 33,
        "252": 34,
        "264": 35,
        "723": 36,
        "2105": 37,
        "2177": 38 
    }
     

In [97]:
# Embedding


class Embedding(nn.Module):
    #poke: 261(+0), moves: 321(+0), items: 38(+0), abilities: 91(+0)
    def __init__(self,d_model=256, feat_dim = 16):
        super().__init__()
        self.d_model=d_model #dimensione complessiva dello spazio degli embeddings 
        #EMBEDDING DELLO STATO s_t
        #Da ripetere per ognuno dei 12 pokemon

        #Features discrete
        self.embed_player=nn.Embedding(num_embeddings = 2, embedding_dim = feat_dim) 
        self.embed_id=nn.Embedding(num_embeddings = 310, embedding_dim = feat_dim) 
        self.embed_type1=nn.Embedding(num_embeddings = 19, embedding_dim = feat_dim) 
        self.embed_type2=nn.Embedding(num_embeddings = 19, embedding_dim = feat_dim) 
        self.embed_ability=nn.Embedding(num_embeddings = 149, embedding_dim = feat_dim) 
        self.embed_item=nn.Embedding(num_embeddings = 39, embedding_dim = feat_dim) 
        self.embed_slot=nn.Embedding(num_embeddings = 5, embedding_dim = feat_dim) 


        #Features continue
        self.embed_stats=nn.Linear(6, feat_dim) 
        self.embed_stats_change=nn.Linear(5, feat_dim)
        self.embed_status=nn.Linear(6, feat_dim)
        self.embed_hp_ratio=nn.Linear(1, feat_dim)
        
        #Mosse
        self.embed_id_move=nn.Embedding(num_embeddings = 322, embedding_dim = feat_dim) 
        self.embed_d_class=nn.Embedding(num_embeddings = 4, embedding_dim = feat_dim) 
        self.embed_move_type=nn.Embedding(num_embeddings = 19, embedding_dim = feat_dim) 
        self.embed_t_class=nn.Embedding(num_embeddings = 17, embedding_dim = feat_dim)  

        #Mosse continue
        self.embed_power=nn.Linear(1,feat_dim) 
        self.embed_priority=nn.Linear(1, feat_dim) 
        self.embed_accuracy=nn.Linear(1, feat_dim) 

        #Campo
        self.embed_current_weather=nn.Embedding(num_embeddings=5,embedding_dim = feat_dim)
        self.embed_speed_modifier=nn.Linear(3,feat_dim)

        #proiezione
        self.state_proj=nn.Linear(in_features=7520, out_features=d_model) 


        #EMBEDDING DELL'AZIONE a_t
        self.embed_player_user=nn.Embedding(num_embeddings = 2, embedding_dim = feat_dim)
        self.embed_slot_user=nn.Embedding(num_embeddings = 3,embedding_dim = feat_dim)
        self.embed_player_target=nn.Embedding(num_embeddings = 2, embedding_dim = feat_dim)
        self.embed_slot_target=nn.Embedding(num_embeddings = 5,embedding_dim = feat_dim)
        self.embed_mega=nn.Embedding(num_embeddings = 2, embedding_dim = feat_dim)
        self.embed_move=nn.Embedding(num_embeddings = 6, embedding_dim = feat_dim)


        self.action_proj=nn.Linear(in_features = 192, out_features=d_model)  #la dimensione di input è il doppio della somma delle dim di embedding (2 mosse)

        #EMBEDDING RICOMPENSA 
        self.embed_reward=nn.Embedding(num_embeddings = 2,embedding_dim=d_model)


        #EMBEDDING DEL TURNO t
        self.embed_turn=nn.Embedding(num_embeddings = 49, embedding_dim=d_model) 

        


    def forward(self, state, move, battlefield, action, reward, turn,padding_mask=None):
        #i nostri token devono essere riorganizzati in tensori con dimensione (batch_size, turn, 12, ...)
        #batch_size=numero di partite, turn=numero di turni in una partita
        #12=numero di pokemon in uno stato

        #move ha dimensione (batch_size,turn,12,4)

        #consideriamo separatamente in tre tensori diversi il token di stato, le mosse e il token del campo,
        #facciamo l'embedding separatamente e poi li concateniamo alla fine

        #l'azione è riorganizzata in un tensore di dimensioni (batch_size,2,...)

        batch_size=state['id'].size(0)

        #EMBEDDING STATO s_t
        #Features discrete
        
        player_emb = self.embed_player(state['player']) 
        id_emb = self.embed_id(state['id'])             
        type1_emb = self.embed_type1(state['type1'])
        type2_emb = self.embed_type2(state['type2'])
        ability_emb = self.embed_ability(state['ability']) 
        item_emb = self.embed_item(state['item'])          
        slot_emb = self.embed_slot(state['slot'])
        
        #Features continue
        stats_emb=self.embed_stats(state['stats'])
        stats_change_emb=self.embed_stats_change(state['stats_change'])
        status_emb=self.embed_status(state['status'])
        hp_ratio_emb=self.embed_hp_ratio(state['hp_ratio'])

        #Mosse
        #move ha dimensione (batch_size,turn,12,4) 
        id_move_emb=self.embed_id_move(move['id'])
        d_class_emb=self.embed_d_class(move['d_class'])
        move_type_emb=self.embed_move_type(move['type'].long())
        t_class_emb=self.embed_t_class(move['t_class'])
        power_emb=self.embed_power(move['power'])

        priority_emb = self.embed_priority(move['priority'].unsqueeze(-1))
        accuracy_emb = self.embed_accuracy(move['accuracy'].unsqueeze(-1))


        #Concatenazione features di un singolo pokemon sull'ultima dimensione
        pokemon_emb=torch.cat([player_emb,id_emb,type1_emb,type2_emb,ability_emb,item_emb,slot_emb,stats_emb,stats_change_emb,status_emb,hp_ratio_emb], dim=-1)
        pokemon_flat=pokemon_emb.view(pokemon_emb.size(0),pokemon_emb.size(1),-1) #stiamo rendendo il tensore una lista piatta per ogni turno

        #Concatenazione features delle mosse di un singolo pokemon sull'ultima dimensione
        move_emb=torch.cat([id_move_emb,move_type_emb,d_class_emb,t_class_emb,power_emb,priority_emb,accuracy_emb], dim=-1)
        move_flat=move_emb.view(move_emb.size(0),move_emb.size(1),-1)

        #EMBEDDING CAMPO
        current_weather_emb=self.embed_current_weather(battlefield['current_weather'])
        speed_modifier_emb=self.embed_speed_modifier(battlefield['speed_modifier'])

        #Concatenazione degli embedding di stato e campo
        full_state=torch.cat([pokemon_flat,move_flat,current_weather_emb,speed_modifier_emb], dim=-1)
        #va proiettato in uno spazio di dimensione d_model
        state_emb=self.state_proj(full_state)

        #EMBEDDING AZIONE a_t e concatenazione
        player_user_emb=self.embed_player_user(action['player_user'])
        slot_user_emb=self.embed_slot_user(action['slot_user']) # slot_user \in {1,2}. so is shifted to {0,1}
        player_target_emb=self.embed_player_target(action['player_target'])
        slot_target_emb=self.embed_slot_target(action['slot_target'])
        mega_emb=self.embed_mega(action['mega'])
        slot_move_emb=self.embed_move(action['move'])
        full_action=torch.cat([player_user_emb, slot_user_emb, player_target_emb, slot_target_emb, mega_emb, slot_move_emb], dim=-1)
        action_flat=full_action.view(full_action.size(0),full_action.size(1),-1) #stiamo rendendo il tensore una lista piatta per ogni turno

        action_emb=self.action_proj(action_flat)


        #EMBEDDING RICOMPENSA
        reward_emb=self.embed_reward(reward)

        #EMBEDDING TURNO t
        turn_emb=self.embed_turn(turn)

        state_emb=state_emb+turn_emb
        action_emb=action_emb+turn_emb
        reward_emb=reward_emb+turn_emb
        seq_length = action_emb.size(1) 
        
        final_inputs = torch.stack([reward_emb, state_emb, action_emb], dim=1).permute(0, 2, 1, 3).reshape(batch_size, 3 * seq_length, self.d_model)

 
        #Attention_mask
        if padding_mask is not None:
            stacked_padding_mask=padding_mask.repeat_interleave(3, dim=1) #ripetiamo l'attention mask per reward, state e action
        else:
            stacked_padding_mask=None

        #La maschera serve per fare in modo che per ogni partita si consideri lo stesso numero di turni. Si aggiungono quindi dei turni fittizi
        #in partite più corte. La maschera assegna 1 ai turni reali e 0 ai turni fittizi
        #La maschera viene creata nel momento in cui si crea il batch di partite e viene passata al modello come input. Viene poi ripetuta per reward, state e action
        
        return final_inputs, stacked_padding_mask

In [98]:
#Self Attention and Transformer Block



class SelfAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=8, max_seq_length=147):
        #147=max_turn(49)*3
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        #Linear projection for query, key, value and output
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)

        self.unifyheads = nn.Linear(d_model, d_model, bias=False)

        #casual mask
        causal_mask=torch.tril(torch.ones(max_seq_length, max_seq_length))
        self.register_buffer("causal_mask", causal_mask)


    def forward(self, x, padding_mask=None): #x is our final_inputs
        batch_size, seq_length, d_model = x.size() 
        #batch_size=numero partite, seq_length=3*numero turni (R,s,a)

        #Linear projection and reshape
        q = self.query(x).view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1,2).contiguous().view(batch_size * self.n_heads, seq_length, self.head_dim)  
        k = self.key(x).view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1,2).contiguous().view(batch_size * self.n_heads, seq_length, self.head_dim)    
        v = self.value(x).view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1,2).contiguous().view(batch_size * self.n_heads, seq_length, self.head_dim)  

        #Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(1, 2)) / math.sqrt(self.head_dim)  

        #Causal mask (tagliamo la causal mask alla lunghezza attuale della sequenza)
        causal_mask = self.causal_mask[:seq_length, :seq_length]

        #Setting to -inf the positions that should be masked
        scores = scores.masked_fill(causal_mask == 0, float('-inf'))


        #Padding mask with dimension (batch_size, seq_length)
        if padding_mask is not None:
            padding_mask = padding_mask.repeat_interleave(self.n_heads, dim=0)  # Repeat the padding mask for each head
            padding_mask = padding_mask.unsqueeze(1)  
            #Setting to -inf the positions of fake turns (padding) that should be masked
            scores = scores.masked_fill(padding_mask == 0, float('-inf')) 

        #Softmax and weighted sum
        attention_weights = F.softmax(scores, dim=-1)
        #Wighted sum of values
        attention_output=torch.bmm(attention_weights, v).view(batch_size, self.n_heads, seq_length, self.head_dim).transpose(1,2).contiguous().view(batch_size, seq_length, d_model)
            

        return self.unifyheads(attention_output)
    
class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=8, dropout=0.0, max_seq_length=147):
        super().__init__()
        self.self_attention = SelfAttention(d_model, n_heads, max_seq_length)
        self.layer_norm1 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(), 
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

        self.layer_norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        # Self-attention
        attention_output = self.self_attention(self.layer_norm1(x), padding_mask)
        x = x + self.dropout(attention_output) #residual connection
        

        # Feed-forward
        feed_forward_output = self.feed_forward(self.layer_norm2(x))
        x = x + feed_forward_output #residual connection

        return x

        



In [99]:
#Decision Transformer


class DecisionTransformer(nn.Module):
    def __init__(self,action_dim=480, d_model=256, n_heads=8, depth=6, max_turn=49):
        super().__init__()
        self.action_dim = action_dim #tutte le possibili azioni (mosse)
        self.d_model=d_model
        self.seq_length=max_turn
        max_seq_length=3*max_turn #max_seq_length is 3 times the number of tokens (R,s,a) for each turn

        #Embedding layer for states, actions, and rewards
        self.token_embedding = Embedding(d_model=d_model)

        # Transformer blocks
        self.tblocks=nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                dropout=0.0,
                max_seq_length=max_seq_length
                ) for _ in range(depth)
        ])
        
        self.predict_action=nn.Linear(d_model, 2*self.action_dim) #per predire il token dell'azione
    
    def forward(self, state, move, battlefield, action, reward, turn, padding_mask=None, legal_action_mask=None):
        #x is a tensor of shape (batch_size, seq_length, d_model)
        batch_size=state['id'].size(0)

        x, stacked_padding_mask=self.token_embedding(state, move, battlefield, action, reward, turn, padding_mask) #embedding layer
        
        for block in self.tblocks:
            x=block(x, padding_mask=stacked_padding_mask) #transformer blocks

        
        x=x.reshape(batch_size, self.seq_length, 3, self.d_model).permute(0,2,1,3) #reshape to (batch_size, 3, seq_length, d_model)

        state_representation=x[:,1] #prendo solo la componente di stato di x. ha dimensione (batch_size, seq_length, d_modele) 
        
        action_preds=self.predict_action(state_representation) #linear layer to get logits for each action
        #va fatta una maschera per le azioni illegali da applicare ad action_preds la maschera mette le azioni illegali di action_preds a -inf
        action_preds=action_preds.view(batch_size, self.seq_length, 2, self.action_dim) 

        if legal_action_mask is not None:
            # legal_action_mask: bool, shape (batch_size, seq_length, action_dim)
            # True = azione lecita. Le due teste condividono lo stesso spazio -> stesso mask per entrambe
            mask = legal_action_mask.unsqueeze(2)  # (batch, seq, 1, action_dim) -> broadcast su dim=2 (le due teste)
            action_preds = action_preds.masked_fill(~mask, float('-inf'))
            
        return F.log_softmax(action_preds, dim=-1) #log softmax over the last dimension (num_tokens)


In [100]:
#Training loop


# Su Kaggle, DEVI salvare in /kaggle/working/ affinché i file 
# vengano conservati alla fine dell'esecuzione
SAVE_DIR = '/kaggle/working/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(epoch, model, optimizer, loss, filename="latest_checkpoint.pth"):
    # Se il modello è wrappato in DataParallel, estraiamo l'architettura base
    if isinstance(model, nn.DataParallel):
        model_state = model.module.state_dict()
    else:
        model_state = model.state_dict()
        
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_state, # Salviamo i pesi puliti
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    path = os.path.join(SAVE_DIR, filename)
    torch.save(checkpoint, path)
    print(f"Checkpoint salvato: {path}")

def load_checkpoint(filepath, model, optimizer):
    if os.path.exists(filepath):
        checkpoint = torch.load(filepath)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint['epoch']
        loss = checkpoint['loss']
        print(f"Checkpoint caricato! Riprendo dall'epoca {epoch} con loss {loss:.4f}")
        return epoch + 1 # Riprendi dall'epoca successiva
    else:
        print("Nessun checkpoint trovato. Inizio da zero.")
        return 0

def train_decision_transformer(model, dataloader_training, dataloader_validation, num_epochs, device, lr=1e-5):
    #Spostiamo il modello sul device (GPU o CPU)
    model.to(device) #model coincide con il nostro Decision Transformer 
    
    # Inizializziamo l'ottimizzatore
    optimizer = AdamW(model.parameters(), lr=lr)
    
    # Usiamo NLLLoss senza riduzione automatica per poter applicare la padding_mask
    criterion = nn.NLLLoss(reduction='none')

    start_epoch = load_checkpoint('/kaggle/working/checkpoints/latest_checkpoint.pth', model, optimizer)

    best_val_loss = float('inf')  # Per tenere traccia della migliore loss di validazione

    for epoch in range(start_epoch,num_epochs):
        model.train()
        total_training_loss = 0.0

        #Training
        for batch in dataloader_training:
            # 1. Spostiamo tutti i dizionari e i tensori sul device corretto
            state = {k: v.to(device) for k, v in batch['state'].items()}
            move = {k: v.to(device) for k, v in batch['move'].items()}
            battlefield = {k: v.to(device) for k, v in batch['battlefield'].items()}
            action = {k: v.to(device) for k, v in batch['action'].items()}
            
            reward = batch['reward'].to(device)
            turn = batch['turn'].to(device)
            padding_mask = batch['padding_mask'].to(device)

            # 1. target_actions è un dizionario, iteriamo per spostare i tensori sul device
            target_dict = {k: v.to(device) for k, v in batch['target_actions'].items()}            
            # target_actions ha shape (batch_size, seq_length, 2) e contiene gli indici reali delle mosse
            # 2. Convertiamo i 6 componenti nell'indice piatto (da 0 a 479).
            # Le dimensioni sono (2, 2, 2, 5, 2, 6). Calcoliamo i moltiplicatori (strides):
            # move: 1
            # mega: 6
            # slot_target: 6 * 2 = 12
            # player_target: 12 * 5 = 60
            # slot_user: 60 * 2 = 120
            # player_user: 120 * 2 = 240

            

            target_actions_flat = (
                target_dict['player_user'] * 240 +
                (target_dict['slot_user'] - 1) * 120 +  # slot_user va da 1 a 2, togliamo 1 per avere 0-1
                target_dict['player_target'] * 60 +
                target_dict['slot_target'] * 12 +
                target_dict['mega'] * 6 +
                target_dict['move']
            ) 
            target_actions_flat = torch.clamp(target_actions_flat, min=0,max=479)
            # 2. Azzeriamo i gradienti
            optimizer.zero_grad()
            
            # 3. Forward pass
            # Passiamo tutti gli argomenti previsti dal forward del DecisionTransformer
            legal_action_mask = batch['legal_action_mask'].to(device)
            log_probs = model(state, move, battlefield, action, reward, turn, padding_mask, legal_action_mask)


            # log_probs ora ha shape (batch_size, seq_length, 2, action_dim)
            # NLLLoss di PyTorch richiede che le classi siano nella seconda dimensione: (N, C, d1, d2, ...)
            # Riorganizziamo il tensore in (batch_size, action_dim, seq_length, 2)
            
            log_probs_transposed = log_probs.permute(0, 3, 1, 2).contiguous()
            
            # 4. Calcolo della Loss
            loss = criterion(log_probs_transposed, target_actions_flat)
            
            # loss ha shape (batch_size, seq_length, 2)
            # Applichiamo la maschera per ignorare i turni fittizi creati nel batch
            # La padding_mask ha shape (batch_size, seq_length), la espandiamo per coprire le 2 azioni
            expanded_mask = padding_mask.unsqueeze(-1).expand(-1, -1, 2)
            
            # Moltiplichiamo la loss per la maschera (azzera la loss dei turni di padding) e facciamo la media
            masked_loss = (loss * expanded_mask).sum() / expanded_mask.sum()
            
            # 5. Backward pass e ottimizzazione
            masked_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_training_loss += masked_loss.item()

        
        avg_training_loss = total_training_loss / len(dataloader_training)

        #VALIDATION
        model.eval() #inizio valutazione
        total_val_loss = 0
        with torch.no_grad():
            for batch in dataloader_validation:
                state = {k: v.to(device) for k, v in batch['state'].items()}
                move = {k: v.to(device) for k, v in batch['move'].items()}
                battlefield = {k: v.to(device) for k, v in batch['battlefield'].items()}
                action = {k: v.to(device) for k, v in batch['action'].items()}
                reward = batch['reward'].to(device)
                turn = batch['turn'].to(device)
                padding_mask = batch['padding_mask'].to(device)
                target_dict = {k: v.to(device) for k, v in batch['target_actions'].items()}

                target_actions_flat = (
                    target_dict['player_user'] * 240 +
                    (target_dict['slot_user'] - 1) * 120 +  # slot_user va da 1 a 2, togliamo 1 per avere 0-1
                    target_dict['player_target'] * 60 +
                    target_dict['slot_target'] * 12 +
                    target_dict['mega'] * 6 +
                    target_dict['move']
                )
                
                target_actions_flat = torch.clamp(target_actions_flat, min=0,max=479)

                legal_action_mask = batch['legal_action_mask'].to(device)
                log_probs = model(state, move, battlefield, action, reward, turn, padding_mask,legal_action_mask)
                log_probs_transposed = log_probs.permute(0, 3, 1, 2)
                loss = criterion(log_probs_transposed, target_actions_flat)
                expanded_mask = padding_mask.unsqueeze(-1).expand(-1, -1, 2)
                masked_loss = (loss * expanded_mask).sum() / expanded_mask.sum()
                
                total_val_loss += masked_loss.item()
                # Process the batch and compute validation loss

        avg_val_loss = total_val_loss / len(dataloader_validation)
        
        save_checkpoint(epoch, model, optimizer, avg_training_loss, "latest_checkpoint.pth")

        # Save the best model based on validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_checkpoint(epoch, model, optimizer, avg_val_loss, "best_model.pth")
            print(f"Nuovo miglior modello salvato all'epoca {epoch+1} con loss di validazione {avg_val_loss:.4f}")

        if epoch % 5 == 0:
            save_checkpoint(epoch, model, optimizer, avg_training_loss, f"checkpoint_epoch_{epoch}.pth")
            print(f"Epoch [{epoch+1}/{num_epochs}] | Loss Media: {avg_training_loss:.4f}")


    return model

In [ ]:
# Main
from torch.utils.data import DataLoader # type: ignore
from sklearn.model_selection import train_test_split # type: ignore



# 4. Passalo alla funzione di train che abbiamo scritto prima
# train_decision_transformer(model, dataloader, num_epochs, device)
def main():
    # 1. Configurazione del Device (GPU se disponibile, altrimenti CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Sto addestrando sul device: {device}")

    # 2. Preparazione dei Dati (Dataset e DataLoader)

    # 1. Trova tutti i file .npy nella cartella dei dati
    lista_files = glob.glob("/kaggle/input/datasets/armandocoppola/reg-m-a/reg_m-A/*.npz")

    #Splittiamo i dati in train, validation e test (80% train, 10% validation, 10% test)
    train_files, temp_files = train_test_split(lista_files, test_size=0.2, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

    # 2. Inizializza il Dataset
    dataset_training = PokemonVGCDataset(file_paths=train_files, max_turn=49)
    dataset_validation = PokemonVGCDataset(file_paths=val_files, max_turn=49)
    dataset_test = PokemonVGCDataset(file_paths=test_files, max_turn=49)

    # 3. Crea il DataLoader del training set
    # num_workers velocizza il caricamento parallelizzando la lettura su CPU
    dataloader_training = DataLoader(
        dataset_training,
        batch_size=32, 
        shuffle=True, 
        num_workers=4, 
        drop_last=True
    )

    #Creiamo il DataLoader del validation set
    dataloader_validation = DataLoader(
        dataset_validation,
        batch_size=32, 
        shuffle=False, 
        num_workers=4, 
        drop_last=False
    )

    # Creiamo il DataLoader del test set
    dataloader_test = DataLoader(
        dataset_test,
        batch_size=32, 
        shuffle=False, 
        num_workers=4, 
        drop_last=False
    )

    

    # 3. Inizializzazione del Modello
    # Richiamiamo il modello usando le dimensioni che avete definito
    model = DecisionTransformer(
        action_dim=480, 
        d_model=256, 
        n_heads=8, 
        depth=4, 
        max_turn=49


    )

    # --- MODIFICA PER IL MULTI-GPU ---
    # Se PyTorch rileva più di una GPU, divide il lavoro in automatico
    if torch.cuda.device_count() > 1:
        print(f"Evviva! Sto usando {torch.cuda.device_count()} GPU!")
        model = nn.DataParallel(model)
    
    # Sposta il modello sul device principale (cuda:0)
    model.to(device)

    # 4. Definizione degli Iperparametri
    EPOCHS = 100
    LEARNING_RATE = 1e-5

    print("Avvio dell'addestramento...")
    
    # 5. LA CHIAMATA ALLA FUNZIONE DI TRAINING
    train_decision_transformer(
        model=model, 
        dataloader_training=dataloader_training, # Passiamo il raggruppatore di dati
        dataloader_validation=dataloader_validation,
        num_epochs=EPOCHS, 
        device=device, 
        lr=LEARNING_RATE
    )
    
    print("Addestramento completato!")

    # 6. Salvataggio del modello addestrato
    # Salviamo solo i "pesi" (state_dict) per poterlo ricaricare in fase di inferenza/gioco
    if isinstance(model, nn.DataParallel):
        torch.save(model.module.state_dict(), "vgc_decision_transformer.pth")
    else:
        torch.save(model.state_dict(), "vgc_decision_transformer.pth")
        
    print("Modello salvato con successo.")

    



if __name__ == "__main__":
    main()


Sto addestrando sul device: cpu
Avvio dell'addestramento...
Nessun checkpoint trovato. Inizio da zero.
